In [3]:
import pandas as pd
import numpy as np


In [7]:
df = pd.read_csv('/Users/lukecheng/OMSCS/CS6603/CS6603_2026_Summer/Porject_3_AI_ML_Part_1/toxicity_per_attribute.csv')

/var/folders/4n/hd5b4m5950sbjp1cnswf_x4h0000gn/T/ipykernel_20358/215964179.py:1: DtypeWarning: Columns (2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/lukecheng/OMSCS/CS6603/CS6603_2026_Summer/Porject_3_AI_ML_Part_1/toxicity_per_attribute.csv')


Step 1: Data Clean up

In [14]:
# get number of row and columns we have
print(df.shape[0])
print(df.shape[1])

76565
52


In [ ]:
# check null value for each column

print(df.isnull().sum().sort_values(ascending=False))

Wiki_ID             2
sikh                1
indian              1
middle eastern      1
chinese             1
japanese            1
christian           1
muslim              1
jewish              1
buddhist            1
catholic            1
protestant          1
taoist              1
american            1
old                 1
older               1
young               1
younger             1
teenage             1
millenial           1
middle aged         1
elderly             1
blind               1
deaf                1
asian               1
canadian            1
male                1
mexican             1
lesbian             1
gay                 1
bisexual            1
transgender         1
trans               1
queer               1
lgbt                1
lgbtq               1
homosexual          1
straight            1
heterosexual        1
paralyzed           1
female              1
nonbinary           1
african             1
african american    1
black               1
white     

In [13]:
# check null value for each row

print(df.isnull().sum(axis=1).sort_values(ascending=False))

52671    51
52672     1
0         0
51040     0
51046     0
         ..
25521     0
25520     0
25519     0
25518     0
76564     0
Length: 76565, dtype: int64


In [15]:
# Drop corrupted row
df = df[df['TOXICITY'] != 52671]

# Assign missing Wiki_ID
df.loc[df['Wiki_ID'].isna(), 'Wiki_ID'] = 52671

In [16]:
# check null value for each row

print(df.isnull().sum(axis=1).sort_values(ascending=False))

0        0
51049    0
51047    0
51046    0
51045    0
        ..
25520    0
25519    0
25518    0
25517    0
76564    0
Length: 76564, dtype: int64


Step 2: 

2.1 Create a new dataframe with rows that is not all FALSE


In [17]:
subgroup_cols = [c for c in df.columns if c not in ['Wiki_ID', 'TOXICITY']]
df_reduced = df[df[subgroup_cols].eq(True).any(axis=1)].copy()
print(f"Original: {len(df)} rows")
print(f"Reduced:  {len(df_reduced)} rows")
print(f"Removed:  {len(df) - len(df_reduced)} rows")

Original: 76564 rows
Reduced:  75700 rows
Removed:  864 rows


### Step 2.2 & 2.3: Encode subgroups and compact into protected class columns

Alphabetical encoding: `0` = FALSE, `1+` = subgroup (A–Z). If multiple subgroups in a class are TRUE, use the **highest** value.

In [18]:
# Alphabetical subgroup -> numeric value mappings (0 = FALSE)
protected_class_mappings = {
    'Gender / Sex': {
        'female': 1, 'male': 2, 'nonbinary': 3, 'trans': 4, 'transgender': 5
    },
    'Sexual Orientation': {
        'bisexual': 1, 'gay': 2, 'heterosexual': 3, 'homosexual': 4, 'lesbian': 5,
        'lgbt': 6, 'lgbtq': 7, 'queer': 8, 'straight': 9
    },
    'Race': {
        'african': 1, 'african american': 2, 'asian': 3, 'black': 4,
        'indian': 5, 'middle eastern': 6, 'white': 7
    },
    'Ethnicity': {
        'european': 1, 'hispanic': 2, 'latina': 3, 'latino': 4, 'latinx': 5
    },
    'National Origin': {
        'american': 1, 'canadian': 2, 'chinese': 3, 'japanese': 4, 'mexican': 5
    },
    'Religion': {
        'buddhist': 1, 'catholic': 2, 'christian': 3, 'jewish': 4, 'muslim': 5,
        'protestant': 6, 'sikh': 7, 'taoist': 8
    },
    'Age': {
        'elderly': 1, 'middle aged': 2, 'millenial': 3, 'old': 4, 'older': 5,
        'teenage': 6, 'young': 7, 'younger': 8
    },
    'Disability': {
        'blind': 1, 'deaf': 2, 'paralyzed': 3
    }
}


def compact_protected_class(df, subgroup_mapping):
  """Combine subgroup boolean columns into one encoded protected-class column."""
  values = np.zeros(len(df), dtype=int)
  for subgroup, code in subgroup_mapping.items():
    is_true = df[subgroup].eq(True)
    values = np.where(is_true, np.maximum(values, code), values)
  return values

In [19]:
# Build compacted dataset: Wiki_ID, TOXICITY, + one column per protected class
df_compacted = df_reduced[['Wiki_ID', 'TOXICITY']].copy()

for class_name, mapping in protected_class_mappings.items():
    df_compacted[class_name] = compact_protected_class(df_reduced, mapping)

print(df_compacted.shape)
df_compacted.head()

(75700, 10)


,Wiki_ID,TOXICITY,Gender / Sex,Sexual Orientation,Race,Ethnicity,National Origin,Religion,Age,Disability
0,0.0,0.096492,0,0,0,0,1,0,0,0
1,1.0,0.017991,4,0,0,0,0,0,0,0
2,2.0,0.150298,0,4,0,0,0,0,0,0
3,3.0,0.065861,0,0,0,0,5,0,0,0
4,4.0,0.667166,0,0,0,0,0,7,0,0


In [20]:
# Verify value ranges per protected class (0 = no subgroup mentioned)
for class_name in protected_class_mappings:
    print(f"{class_name}: min={df_compacted[class_name].min()}, max={df_compacted[class_name].max()}, unique={sorted(df_compacted[class_name].unique())}")

Gender / Sex: min=0, max=5, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Sexual Orientation: min=0, max=9, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Race: min=0, max=7, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]
Ethnicity: min=0, max=5, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
National Origin: min=0, max=5, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Religion: min=0, max=8, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Age: min=0, max=8, unique=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Disability: min=0, max=3, unique=[np.int64(0), np.int64(1), n